# Entrenamiento: cuello arriba vs neutro (esplenio de la cabeza izquierdo)

**Qué hace este notebook**, celda por celda:
1. Trae el repositorio (código + datos de entrenamiento) desde GitHub.
2. Lee `data/metadata.csv` y usa solo las grabaciones de **entrenamiento** que pasaron la calidad.
3. Filtra cada grabación con **SinArm** (el mismo filtro del Arduino) y la corta en ventanas de **100 ms**.
4. Compara pisos y modelos **dejando una sesión fuera** (electrodos recolocados), como la prueba final.
5. Entrena el modelo elegido con todas las sesiones de entrenamiento y lo guarda con su versión.
6. **Al final y una sola vez**: abre las sesiones de prueba desde **Azure** y da la exactitud.

**Antes de correrlo en Colab:** Secrets (icono de llave, a la izquierda) →
`GITHUB_TOKEN` (token de GitHub con permiso de lectura del repo) y, solo para la celda 8,
`AZURE_STORAGE_CONNECTION_STRING`. Ninguna clave se escribe en el notebook.

In [ ]:
# Celda 1 - traer el repositorio
import os, sys, subprocess
from pathlib import Path

REPO = "Aleja246/DataBase_sEMD_adq"
EN_COLAB = "google.colab" in sys.modules
USAR_SIMULADO = os.environ.get("EMG_SIMULADO", "0") == "1"   # True solo para ensayar sin datos reales

if EN_COLAB:
    from google.colab import userdata
    RAIZ = Path("/content/DataBase_sEMD_adq")
    url = f"https://{userdata.get('GITHUB_TOKEN')}@github.com/{REPO}.git"
    if RAIZ.exists():
        r = subprocess.run(["git", "-C", str(RAIZ), "pull", "-q", url, "main"], capture_output=True)
    else:
        r = subprocess.run(["git", "clone", "-q", "--depth", "1", url, str(RAIZ)], capture_output=True)
        subprocess.run(["git", "-C", str(RAIZ), "remote", "set-url", "origin",
                        f"https://github.com/{REPO}.git"])        # el token no se queda guardado
    if r.returncode != 0:
        raise RuntimeError("No se pudo traer el repo: revisen el secreto GITHUB_TOKEN.")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "azure-storage-blob", "xlsxwriter"])
else:
    RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

sys.path.insert(0, str(RAIZ / "code"))
print("Repositorio:", RAIZ, "| simulado:", USAR_SIMULADO)

In [ ]:
# Celda 2 - configuración y metadata (solo entrenamiento)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from emg_cuello.config import cargar_config, carpeta
from emg_cuello.datos import leer_metadata, leer_registro, ruta_de_fila
from emg_cuello.procesar import senal_sinarm, tabla_ventanas, filas_utiles
from emg_cuello.filtros import adc_a_voltios
from emg_cuello import modelos

cfg = cargar_config()
CARPETA = carpeta(cfg, "simulated") if USAR_SIMULADO else None
RUTA_META = (CARPETA / "metadata.csv") if USAR_SIMULADO else RAIZ / "data" / "metadata.csv"
FIGURAS = carpeta(cfg, "outputs") if USAR_SIMULADO else RAIZ / "figures"
FIGURAS.mkdir(parents=True, exist_ok=True)

meta = leer_metadata(RUTA_META)
train = meta[meta["set"] == "train"]
tabla_sesiones = (train.assign(kept=train["trials_rejected"] == "0")
                  .groupby("session").agg(recordings=("file_name", "count"), kept=("kept", "sum")))
print(f"Entrenamiento: {len(train)} grabaciones, {int(tabla_sesiones['kept'].sum())} aceptadas")
tabla_sesiones

In [ ]:
# Celda 3 - una grabación con sus fases marcadas (figura para el README)
fila = filas_utiles(meta, "train").iloc[0]
ruta = (CARPETA / fila["file_name"]) if USAR_SIMULADO else ruta_de_fila(cfg, fila)
df = leer_registro(ruta)
t = df["time_ms"] / 1000
v = adc_a_voltios(df["emg_adc"], cfg["vref_v"], cfg["adc_bits"])
y = senal_sinarm(df, cfg)
colores = {"rest": "tab:green", "warning": "gold", "move": "tab:red", "between": "tab:blue"}
fig, ax = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
ax[0].plot(t, v, lw=0.4, color="k"); ax[0].set_ylabel("raw (V)")
ax[1].plot(t, 1000 * y, lw=0.4, color="k"); ax[1].set_ylabel("SinArm (mV)")
for fase, g in df.groupby((df["phase"] != df["phase"].shift()).cumsum()):
    for a in ax:
        a.axvspan(t[g.index[0]], t[g.index[-1]], color=colores[g["phase"].iloc[0]], alpha=0.15)
ax[1].set_xlabel("time (s)")
ax[0].set_title(f"{fila['file_name']}: rest (green), warning (yellow), neck up (red), return (blue)")
fig.tight_layout(); fig.savefig(FIGURAS / "one_recording_marked.png", dpi=120); plt.show()

In [ ]:
# Celda 4 - ventanas de 100 ms con sus 6 características
tabla = tabla_ventanas(meta, cfg, "train", carpeta=CARPETA)
print(f"{len(tabla)} ventanas de entrenamiento")
tabla.groupby(["session", "phase"]).size().unstack()

In [ ]:
# Celda 5 - ¿se separan reposo y arriba? (figura para DECISIONS.md)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for etiqueta, nombre, c in [(0, "rest (neutral)", "tab:green"), (1, "neck up", "tab:red")]:
    g = tabla[tabla["label"] == etiqueta]
    ax[0].scatter(np.log10(1000 * g["rms"]), g["mnf"], s=4, alpha=0.4, color=c, label=nombre)
    ax[1].hist(np.log10(1000 * g["rms"]), bins=60, alpha=0.5, color=c, label=nombre)
ax[0].set_xlabel("log10 RMS (mV)"); ax[0].set_ylabel("MNF (Hz)"); ax[0].legend()
ax[1].set_xlabel("log10 RMS (mV)"); ax[1].set_ylabel("windows"); ax[1].legend()
fig.tight_layout(); fig.savefig(FIGURAS / "two_features.png", dpi=120); plt.show()

In [ ]:
# Celda 6 - pisos y modelos, dejando una sesión fuera en cada vuelta
resumen, por_sesion, confusiones = modelos.validar_por_sesion(tabla)
display(resumen.style.format({"mean": "{:.1%}", "std": "{:.1%}", "min": "{:.1%}"}))
fig, axs = plt.subplots(1, len(confusiones), figsize=(3.2 * len(confusiones), 3))
for a, (nombre, cm) in zip(axs, confusiones.items()):
    a.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            a.text(j, i, cm[i, j], ha="center", va="center")
    a.set_xticks([0, 1], ["rest", "up"]); a.set_yticks([0, 1], ["rest", "up"])
    a.set_xlabel("model said"); a.set_ylabel("true"); a.set_title(nombre, fontsize=7)
fig.tight_layout(); plt.show()

In [ ]:
# Celda 7 - elegir el modelo, entrenarlo con TODO el entrenamiento y guardarlo
MODELO_ELEGIDO = None   # escriban aquí el nombre de la tabla de la celda 6; None = el de mayor promedio
if MODELO_ELEGIDO is None:
    MODELO_ELEGIDO = resumen.sort_values("mean", ascending=False).iloc[0]["model"]
    print("No eligieron: uso el de mayor promedio. Escriban su elección y la razón en DECISIONS.md.")
modelo, columnas = modelos.entrenar_final(tabla, MODELO_ELEGIDO)
(RAIZ / "models").mkdir(exist_ok=True)
sufijo = "_simulado" if USAR_SIMULADO else ""
ficha = modelos.guardar(modelo, columnas, MODELO_ELEGIDO, resumen, len(tabla),
                        RAIZ / "models" / f"modelo{sufijo}.joblib",
                        RAIZ / "models" / f"modelo{sufijo}.json", RAIZ)
ficha

## Celda 8: prueba final, **una sola vez**

Abre las sesiones de prueba (S11 a S14, electrodos nuevos, guardadas solo en Azure).
Cámbienla a `ABRIR_TEST = True` **solo** cuando el modelo ya esté elegido y guardado.
El resultado queda en `results/test_result.json`. Si ya existe, la celda se detiene:
abrir la prueba dos veces se reporta en el README.

In [ ]:
# Celda 8 - prueba final (una sola vez)
import json, datetime
from sklearn.metrics import accuracy_score, confusion_matrix
ABRIR_TEST = False
REPETIR_TEST = False        # True solo si de verdad necesitan abrirla otra vez (y lo reportan)

ruta_resultado = RAIZ / "results" / ("test_result_simulado.json" if USAR_SIMULADO else "test_result.json")
if not ABRIR_TEST:
    print("Prueba sellada. Cambien ABRIR_TEST = True cuando el modelo esté elegido.")
elif ruta_resultado.exists() and not REPETIR_TEST:
    print("La prueba ya se abrió una vez:", json.loads(ruta_resultado.read_text()))
else:
    if USAR_SIMULADO:
        carpeta_test, meta_test = CARPETA, meta
    else:
        from emg_cuello import nube
        cont = nube.contenedor(cfg["azure_container"])
        carpeta_test = carpeta(cfg, "azure_download") / cfg["participant"]
        ruta_meta_test, _ = nube.bajar_participante(cont, cfg["participant"], carpeta_test)
        meta_test = leer_metadata(ruta_meta_test)
    tabla_test = tabla_ventanas(meta_test, cfg, "test", carpeta=carpeta_test)
    pred = modelo.predict(tabla_test[columnas].to_numpy(dtype=float))
    acc = accuracy_score(tabla_test["label"], pred)
    cm = confusion_matrix(tabla_test["label"], pred, labels=[0, 1])
    resultado = {"model": MODELO_ELEGIDO, "accuracy": round(float(acc), 4),
                 "confusion_matrix_rows_true_cols_pred": cm.tolist(),
                 "test_windows": int(len(tabla_test)),
                 "test_sessions": sorted(tabla_test["session"].unique().tolist()),
                 "opened_at": datetime.datetime.now().isoformat(timespec="seconds"),
                 "git_commit": modelos.commit_actual(RAIZ)}
    ruta_resultado.parent.mkdir(exist_ok=True)
    ruta_resultado.write_text(json.dumps(resultado, indent=2))
    print(f"Exactitud en sesiones nunca vistas: {acc:.1%}  (meta del curso: más de 75 %)")
    print(cm)